In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [2]:
parsed_args = RayTracing.parse_commandline()
    
parsed_args["scene-number"] = 1

1

In [3]:
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

Random.TaskLocalRNG()

In [4]:
I, scene = RayTracing.build_scene(parsed_args)
wbounds = RayTracing.world_bounds(scene.b)


There are 91 objects in the scene, building BVH
  0.027345 seconds (86.00 k allocations: 6.138 MiB, 97.80% compilation time)
Done building BVH
Using 5 samples per pixel
There are 42 lights in the scene


Main.RayTracing.Bounds3([-978.8225099390856, -0.0001, -992.9646455628165], [300.0, 265.0, 300.0])

In [7]:
max_voxels = 64

diag = RayTracing.diagonal(wbounds)
bmax = maximum(diag)
n_voxels = max.(1, Int.(round.(diag / bmax * max_voxels)))

3-element StaticArraysCore.SVector{3, Int64} with indices SOneTo(3):
 63
 13
 64

In [ ]:
p = RayTracing.Pnt3(0, 0, 0)
offset = RayTracing.offset(wbounds, p)
p_i = clamp.(floor.(offset .* n_voxels), 0, n_voxels .- 1)


p0 = p_i ./ n_voxels
p1 = (p_i .+ 1) ./ n_voxels
voxel_bounds = RayTracing.Bounds3(
    RayTracing.lerp(p0, wbounds),
    RayTracing.lerp(p1, wbounds)
)

n_samples = 128
light_contribution = zeros(Float64, length(scene.lights))

for i in 1:n_samples
    po = RayTracing.lerp(
        RayTracing.Pnt3(
            RayTracing.radical_inverse(0, i),
            RayTracing.radical_inverse(1, i),
            RayTracing.radical_inverse(2, i)
        ), 
        voxel_bounds
    )
    intr = RayTracing.Interaction()
    intr.p = po

    u = RayTracing.Pnt2(
        RayTracing.radical_inverse(3, i),
        RayTracing.radical_inverse(4, i)
    )

    for j in 1:lenght(scene.lights)
        radiance, _, pdf_val, _, _, _ = RayTracing.sample_li(scene.lights[j], intr, u)
        if pdf_val > 0.0
            light_contribution[i] += RayTracing.y_spectrum(radiance) / pdf_val
        end
    end
end

42-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0